# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [15]:
# 1 - making sure the setup is working
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [16]:
import dask.dataframe as dd
import time
import pandas as pd
import sys


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [17]:
import os
from glob import glob

# 2 - loading the environment variables
os.getenv("PRICE_DATA")

'../../05_src/data/prices/'

In [ ]:
# loading logger and utils/data paths just in case
sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)

In [19]:
PRICE_DATA = os.getenv("PRICE_DATA")
import shutil

In [20]:
PRICE_DATA

'../../05_src/data/prices/'

In [ ]:
# i really hope that eventually i'll learn to write the below cells completely on my own without templates ._.
temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")
shutil.rmtree(csv_dir, ignore_errors=True)
stock_csv = os.path.join(csv_dir, "stock_px.csv")
os.makedirs(csv_dir, exist_ok=True)

In [22]:
parquet_dir = os.path.join(temp, "parquet")
shutil.rmtree(parquet_dir, ignore_errors=True)
os.makedirs(parquet_dir, exist_ok=True)

In [23]:
import random

stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))

random.seed(42)
stock_files = random.sample(stock_files, 60)

dt_list = []
for s_file in stock_files:
    _logs.info(f"Reading file: {s_file}")
    dt = pd.read_csv(s_file).assign(
        source = os.path.basename(s_file),
        ticker = os.path.basename(s_file).replace('.csv', ''),
        Date = lambda x: pd.to_datetime(x['Date'])
    )
    dt_list.append(dt)
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)

2025-09-26 15:20:25,581, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\TNC.csv
2025-09-26 15:20:25,606, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CBB.csv
2025-09-26 15:20:25,622, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ALDX.csv
2025-09-26 15:20:25,631, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\GLADD.csv
2025-09-26 15:20:25,638, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\FIXX.csv
2025-09-26 15:20:25,641, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ETJ.csv
2025-09-26 15:20:25,649, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CMCTP.csv
2025-09-26 15:20:25,652, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\BWG.csv
2025-09-26 15:20:25,658, 3781317398.py, 10, INFO, Reading file: ../../05_src/data/prices_csv/stocks\VIAC.csv
2025-09-26 15:20:25,6

In [24]:
ticker_dt = stock_prices[stock_prices['ticker'] == 'TNC']
ticker_dt.Date.dt.year.unique() # add variable Year

array([1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983,
       1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994,
       1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005,
       2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016,
       2017, 2018, 2019, 2020], dtype=int32)

In [25]:
stock_prices

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker
0,1973-02-22,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
1,1973-02-23,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
2,1973-02-26,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
3,1973-02-27,4.25,4.25,4.25,4.25,0.507389,0.0,TNC.csv,TNC
4,1973-02-28,4.25,4.25,4.25,4.25,0.507389,0.0,TNC.csv,TNC
...,...,...,...,...,...,...,...,...,...
239654,2020-03-26,9.97,11.59,9.65,11.46,11.460000,22494500.0,KEY.csv,KEY
239655,2020-03-27,10.72,11.69,10.70,11.20,11.200000,20794800.0,KEY.csv,KEY
239656,2020-03-30,11.04,11.24,10.38,10.79,10.790000,15175400.0,KEY.csv,KEY
239657,2020-03-31,10.68,10.86,10.12,10.37,10.370000,15997000.0,KEY.csv,KEY


In [ ]:
# not sure if this was the problem. I'd get an error before I added this
# i'll leave the DataManager be for now
from stock_prices.data_manager import DataManager
dm = DataManager()
dm.price_dir

'../../05_src/data/prices/'

In [27]:
from glob import glob

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = (dd
         .read_parquet(parquet_files)
         .set_index("ticker"))

In [28]:
print(parquet_files)

['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2009\\part.0.parque

In [29]:
# join and read the parquet files

dd_px = (dd
         .read_parquet(parquet_files)
         .set_index("ticker"))

In [30]:
# lazy execution
dd_px

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=90,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ALDX,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


In [34]:
dd_px.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
ticker,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001
...,...,...,...,...,...,...,...,...,...
ZIXI,2005-06-27,3.15,3.15,3.03,3.04,3.040000,182500.0,ZIXI.csv,2005
ZIXI,2005-06-28,3.12,3.12,3.04,3.06,3.060000,120800.0,ZIXI.csv,2005
ZIXI,2005-06-29,3.09,3.24,3.02,3.12,3.120000,326200.0,ZIXI.csv,2005


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [31]:
# 
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = (dd
         .read_parquet(parquet_files)
         .set_index("ticker"))

In [32]:
# Says meta is not specified. Not sure how to add it,
# shifting the data down by 1 day and applying the lambda function
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)


C:\Users\Paul\AppData\Local\Temp\ipykernel_11424\80079922.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [35]:
# calculating the returns
dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

In [36]:
# returns - lazy execution
dd_rets

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
npartitions=90,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...


In [37]:
# computing the returns
dd_rets.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
ticker,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,-0.010547
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,-0.000666
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,-0.009333
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,0.006057
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2005-06-27,3.15,3.15,3.03,3.04,3.040000,182500.0,ZIXI.csv,2005,3.08,-0.012987
ZIXI,2005-06-28,3.12,3.12,3.04,3.06,3.060000,120800.0,ZIXI.csv,2005,3.04,0.006579
ZIXI,2005-06-29,3.09,3.24,3.02,3.12,3.120000,326200.0,ZIXI.csv,2005,3.06,0.019608


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# converting to pandas df and adding moving average. Had to research the code ._.
# left the name of the feature as just returns_ma_10
# (starts when we have at least 10 days worth of price data)
dd_rets_ma = dd_rets.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(returns_ma_10 = x['Returns'].rolling(10).mean())
)
dd_rets_ma

C:\Users\Paul\AppData\Local\Temp\ipykernel_11424\2034991465.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_rets_ma = dd_rets.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns,returns_ma_10
npartitions=90,,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...


In [40]:
dd_rets_ma.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns,Returns_MA_10
ticker,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,-0.010547,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,-0.000666,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,-0.009333,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,0.006057,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2005-06-27,3.15,3.15,3.03,3.04,3.040000,182500.0,ZIXI.csv,2005,3.08,-0.012987,-0.000228
ZIXI,2005-06-28,3.12,3.12,3.04,3.06,3.060000,120800.0,ZIXI.csv,2005,3.04,0.006579,-0.002521
ZIXI,2005-06-29,3.09,3.24,3.02,3.12,3.120000,326200.0,ZIXI.csv,2005,3.06,0.019608,-0.001834


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

**It was fine as the dataset is not too large. Pandas are a powerful tool and have moving average return. I don't know how I would've coded it otherwise yet.**
+ Would it have been better to do it in Dask? Why?

**Dask would be faster, plus we have data stored in Parquet already**

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.